<a href="https://colab.research.google.com/github/JohnnySolo/Data-Analysis-Project---Hybrid-Cyber-Threat-Detection/blob/main/01_Data_Ingestion_and_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Ingestion and ETL (Extract, Transform, Load)

This first notebook is purely for connecting to the AWS S3 bucket via the CLI, extracting the data, and using PySpark to clean, filter, and prepare the dataset.

The final step of this notebook is saving the clean, processed data into a new file or database layer.

Our data source: The `CSE-CIC-IDS2018` dataset.
This is a massive, realistic cyber defense dataset created by the Canadian Institute for Cybersecurity, and it is natively hosted on a public AWS S3 bucket, so we won't need an AWS account, a credit card, or a password to use it.

## Brick 1: Install our project Big Data Engine (PySpark)

Google Colab is a Python environment, but it doesn't have PySpark installed by default. So my first step here is first to install the engine that will allow us to process large datasets efficiently.

In [ ]:
!pip install pyspark

## Brick 2: Look inside the AWS Cloud Storage

Next, we need to see what files are actually sitting in the AWS S3 bucket - To do this, we will use the AWS CLI (Command Line Interface).

This tool is already pre-installed in Google Colab, which is highly convenient.

Moreover, because the CIC made this a public dataset on AWS, we just have to add a special tag (`--no-sign-request`) to our command so AWS knows we are accessing a public directory.

In [ ]:
# Install the AWS Command Line Interface (CLI)
!pip install awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: docutils
    Found existing installation: docutils 0.21.2
    Uninstalling docutils-0.21.2:
      Successfully uninstalled docutils-0.21.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [ ]:
# Look inside the AWS Cloud Storage
!aws s3 ls s3://cse-cic-ids2018/ --no-sign-request

                           PRE Original Network Traffic and Log data/
                           PRE Processed Traffic Data for ML Algorithms/


## Brick 3: Extract the Data from the S3 bucket (Download to Colab)

We want to copy one part of the csv files in the "Processed" folder inside the S3 bucket. This folder contains CSV files where researchers have already run the raw traffic through an extraction tool to pull out over 80 numerical features (such as packet lengths, flag counts, and request time distributions).

Those CSV files from the AWS cloud can be copied directly into the Colab notebook temporary local storage.

In [ ]:
!aws s3 ls "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/" --no-sign-request

2018-10-11 16:02:25          0 
2018-10-11 16:02:49  352368373 Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:03:10  333723605 Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:03:33  382840456 Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:03:59 4054925350 Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:08:38  107842858 Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:08:48  375945899 Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:09:20  382636202 Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:09:44  358223333 Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:10:12  328893673 Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
2018-10-11 16:10:33  209249758 Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


The files mentioned in the output above are our exact options.

The researchers who created the CSE-CIC-IDS2018 dataset executed different cyberattacks on specific days of the week, which is why the files are named by date.

Here is the "cheat sheet" (based on [Kaggle](https://https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv) reference on it) mapping those files to the specific attacks they contain:

* Wednesday-14-02-2018: FTP-BruteForce, SSH-Bruteforce.
* Thursday-15-02-2018: DoS attacks (GoldenEye, Slowloris).
* Friday-16-02-2018: DoS attacks (SlowHTTPTest, Hulk).
* Thuesday-20-02-2018: DDoS attacks (LOIC-HTTP).
* Wednesday-21-02-2018: DDoS attacks (LOIC-UDP, HOIC).
* Thursday-22-02-2018 & Friday-23-02-2018: Web Attacks (Brute Force Web, Brute Force XSS, SQL Injection).
* Wednesday-28-02-2018: Infiltration attacks.
* Thursday-01-03-2018: Infiltration attacks.
* Friday-02-03-2018: Botnet attacks.

My choice is:
* Wednesday-28-02-2018: Infiltration attacks
* Thursday-01-03-2018: Infiltration attacks.

Explanation:

1. Common use by ciber attackers on High-Tech companies.
  
      State-sponsored actors from countries like Iran and Russia, as well as highly sophisticated proxies, frequently rely on Advanced Persistent Threats (APTs) to silently infiltrate enterprise networks [[1]](https://https://www.cohnreznick.com/insights/protecting-us-organizations-from-nation-state-cyber-threats).
      
      In a data-driven high-tech company, the goal of these attackers is rarely just to break things; it is to silently steal valuable intellectual property, customer databases, or AI algorithms [[2]](https://https://www.imperva.com/learn/application-security/apt-advanced-persistent-threat/).
      
      Because these attackers actively try to blend in with normal daily network traffic, traditional rule-based security software often misses them[[3]](https://https://www.netwitness.com/blog/network-forensics-in-cybersecurity/).

      Detecting an infiltration requires finding a "needle in a haystack" by spotting subtle behavioral deviations over time[[4]](https://https://searchinform.com/articles/cybersecurity/cyber-threats/type/data-exfiltration/). This is the exact problem our unsupervised anomaly detection model (Isolation Forest) will be designed to solve.

2. Botnets and DDoS Attacks

      Non-official country representatives and hacktivist groups targeting Israeli infrastructure frequently utilize massive botnets to launch Distributed Denial of Service (DDoS) attacks. Their goal is to overwhelm servers and disrupt the services of tech companies[[5]](https://https://www.fortinet.com/resources/cyberglossary/ddos-attack#:~:text=A%20DDoS%20attack%20aims%20to,them%20inaccessible%20to%20or%20useless).
      
      Training an AI model to rapidly identify patterns of botnet traffic so it can be filtered out from legitimate user traffic is a critical, real-world skill for companies trying to maintain their application's uptime[[6]](https://https://deviceauthority.com/ai-in-iot-security-how-machine-learning-prevents-botnet-attacks-like-eleven11bot/).




In [ ]:
import os

# Ensure the current directory exists and is writable for downloads
current_directory = '/content/'

# Correctly copy files to the current directory
!aws s3 cp "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv" {current_directory} --no-sign-request
!aws s3 cp "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv" {current_directory} --no-sign-request

download: s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv to ./Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
download: s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv to ./Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


## Brick 4: Fire up PySpark and Read the Data

Now that the data has been extracted from the cloud and is sitting in your Colab environment, it's time to act like a Data Scientist. We will start a Spark Session and load the CSV file into a "Spark DataFrame".

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.appName("Cyber_ETL").getOrCreate()

# Load the correct two CSVs
df_infil = spark.read.csv("Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv", header=True, inferSchema=True)
df_botnet = spark.read.csv("Friday-02-03-2018_TrafficForML_CICFlowMeter.csv", header=True, inferSchema=True)

# Merge (union) them together into one dataset
df_raw = df_infil.union(df_botnet)

# Print the total number of rows to see how big our data is
print("Total rows before cleaning:", df_raw.count())

Total rows before cleaning: 1661679


## Brick 5: Transform Data (Basic Preprocessing)

The CSE-CIC-IDS2018 dataset is famous in the data science community for being incredibly realistic, which means it is also "dirty". If we feed this raw data into an ML models later our models could crash.

The benchmark rules for cleaning data like this are:

* **Drop Duplicates**: The network sensors could accidentally record millions of duplicate rows. We must remove them to prevent our model from becoming biased.

* **Drop Missing Values (NaNs)**: Network packets could occasionally drop, leaving empty cells.

* **Drop "Infinity" Values**: Sometimes, calculating "Packets per Second" results in dividing by zero, which creates an "Infinity" value. Machine learning algorithms cannot calculate math on infinity. Therefore, we'll drop those observations too.

In [ ]:
# 1. Drop duplicate rows
df_clean = df_raw.dropDuplicates()

# 2. Drop rows with any missing values (NaN/Null)
df_clean = df_clean.dropna()

# 3. Filter out rows where 'Flow Packets/s' or 'Flow Bytes/s' equals Infinity
# (We cast them to strings briefly to check for the word 'Infinity')
from pyspark.sql.functions import col
df_clean = df_clean.filter(
    (col("Flow Byts/s").cast("string")!= "Infinity") &
    (col("Flow Pkts/s").cast("string")!= "Infinity")
)

print("Total rows after cleaning:", df_clean.count())

Total rows after cleaning: 1641268


## Brick 6: Load the Clean Data (Extract the final file)

Finally, we need to save this clean dataframe so we could use it in our upcoming Exploratory Data Analysis (EDA) phase (in the 2nd notebook).

We will save it as a Parquet file, which is highly compressed and the industry standard format for Big Data processing.

In [ ]:
# Save the clean dataframe as a Parquet file
df_clean.write.mode("overwrite").parquet("/content/clean_cyber_data.parquet")
print("ETL Pipeline Complete! Data saved to Parquet.")

ETL Pipeline Complete! Data saved to Parquet.
